# 03 Describe sections

In [1]:
cd ../..

c:\Users\tangr\GitHub\pc-centrelines


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from os.path import exists, split, join
from os import makedirs
from tqdm import tqdm

import base.geometry as geom

from base.utils import list_all_centrelines, return_centreline_parametrisation



C:\Users\tangr\AppData\Roaming\Python\Python310\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# change here the path to the repository of karst catalogue
#data_repository = "F:/ScanLeica/data_paper/KarstConduitCatalogue" 
data_repository = "D:/KarstConduitCatalogue" 

centreline_filepaths = list_all_centrelines(data_repository)

In [4]:
idx = 3
centreline_filepath = centreline_filepaths[idx]
assert exists(centreline_filepath)
centreline = np.loadtxt(centreline_filepath, skiprows=1)

In [5]:
sections_directory = join(split(centreline_filepath)[0], "sections")

In [6]:
interpolated_centreline, curvilinear_coords, normed_vectors = return_centreline_parametrisation(centreline, dt = 0.5)

In [7]:
sections = [np.loadtxt(join(sections_directory, f"s{c}.txt"), skiprows=1) for c in range(len(normed_vectors))]

In [8]:
sections2d = [geom.to2D(section, nP) for section, nP in zip(sections, normed_vectors)]

In [9]:
# initialise a list of dictionaries 

descriptors = []

for section, curvilinear_coord in tqdm(zip(sections2d, curvilinear_coords), total=len(sections2d)):
    try:
        description = geom.describe(section)
        description["curvilinear_coord"] = curvilinear_coord

        descriptors.append(description)
    except ValueError:
        pass

100%|██████████| 447/447 [00:04<00:00, 95.68it/s] 


In [ ]:
# describe some parameters. 


square = np.array([[0,0], [1, 0.],[1,1.],[0,1]])
square2 = np.array([[0,-1/np.sqrt(2)], [1/np.sqrt(2), 0.],[0,1/np.sqrt(2)],[-1/np.sqrt(2),0]])
section_circ = np.array([np.cos(np.linspace(-np.pi, np.pi)), np.sin(np.linspace(-np.pi, np.pi))]).T
section_ellipse = np.array([2*np.cos(np.pi / 4 + np.linspace(-np.pi, np.pi)), 1/2*np.sin(np.linspace(-np.pi, np.pi))]).T
section = np.array([[0,0], [0.5, 0.4],[1,0.2],[1,1],[0.5,1.5],[0, 1],[-0.5, 0.5]])


In [9]:
descriptors_filepath = join(split(centreline_filepath)[0], "section_descriptors.txt")

descriptor_keys = [elem for elem in descriptors[0].keys() if elem != "cog" and elem !="ellipse_area" and elem !="ellipse_axes" and elem !="ellipse_eigenvalues"]
m = len(descriptors)
n = len(descriptor_keys)
data = np.zeros((m,n))

header = ",".join(descriptor_keys)
print(header)
# cycle over each descriptor and store to numpy array. 
for c, key in enumerate(descriptor_keys):
    data[:, c] =  [elem[key] for elem in descriptors]

# save the descriptor data 
np.savetxt(descriptors_filepath, data, header = header)

area,ellipse_aspect_ratio,ellipse_rotation,perimeter,solidity,hydraulic_diameter,max_chord_length,rmax,rmin,r_average,circularity,circularity2,gravelius_compactness,circularity3,deviation_from_inscribed_disk,deviation_from_circumscribed_disk,elongation,isoperimetric_deficiency,morton_spreading_index,circular_variance,is_closed,curvilinear_coord


Now we can describe the rest of the sections using a filling up curve. 

In [10]:
n = 50
fill_fractions = np.linspace(0, 1, n)

In [11]:
for k, section2d in tqdm(enumerate(sections2d), total= len(sections2d)):
    
    areas = np.zeros(n)
    wetted_perimeters = np.zeros(n)

    for c,fill_fraction in enumerate(fill_fractions):
        areas[c] = geom.get_full_area(section2d, fill_height=fill_fraction)

        filled_areas = geom.return_below_fill_height(section2d, fill_height=fill_fraction)
        wetted_perimeter = 0
        
        if fill_fraction != 1:
            for filled_area in filled_areas:
                wetted_perimeter += geom.compute_ngon_wetted_perimeter(filled_area)
        else:
            wetted_perimeter = geom.compute_ngon_perimeter(section2d)

        wetted_perimeters[c] = wetted_perimeter
    
    data = np.vstack([fill_fractions, areas, wetted_perimeters]).T

    fill_curves_filepath = join(sections_directory, f"fill_curve_s{k}.txt")

    np.savetxt(fill_curves_filepath, data, header = "h, area, wetted_perimeter")

100%|████████████████████████████████████████████████████████████████████████████████| 447/447 [02:40<00:00,  2.79it/s]


In [ ]:
run_main_loop = True
n = 50
fill_fractions = np.linspace(0, 1, n)

if run_main_loop == True:

    for idx in range(len(descriptors_filepath)):
        try:
            
            centreline_filepath = centreline_filepaths[idx]
            assert exists(centreline_filepath)
            centreline = np.loadtxt(centreline_filepath, skiprows=1)

            sections_directory = join(split(centreline_filepath)[0], "sections")

            interpolated_centreline, curvilinear_coords, normed_vectors = return_centreline_parametrisation(centreline)

            sections = [np.loadtxt(join(sections_directory, f"s{c}.txt"), skiprows=1) for c in range(len(normed_vectors))]

            sections2d = [geom.to2D(section, nP) for section, nP in zip(sections, normed_vectors)]

            # initialise a list of dictionaries 

            descriptors = []

            for section, curvilinear_coord in tqdm(zip(sections2d, curvilinear_coords), total=len(sections2d)):
                try:
                    description = geom.describe(section)
                    description["curvilinear_coord"] = curvilinear_coord

                    descriptors.append(description)
                except ValueError:
                    pass
            
            for k, section2d in tqdm(enumerate(sections2d), total= len(sections2d)):
    
                areas = np.zeros(n)
                wetted_perimeters = np.zeros(n)

                for c,fill_fraction in enumerate(fill_fractions):
                    areas[c] = geom.get_full_area(section2d, fill_height=fill_fraction)

                    filled_areas = geom.return_below_fill_height(section2d, fill_height=fill_fraction)
                    wetted_perimeter = 0
                    
                    if fill_fraction != 1:
                        for filled_area in filled_areas:
                            wetted_perimeter += geom.compute_ngon_wetted_perimeter(filled_area)
                    else:
                        wetted_perimeter = geom.compute_ngon_perimeter(section2d)

                    wetted_perimeters[c] = wetted_perimeter
                
                data = np.vstack([fill_fractions, areas, wetted_perimeters]).T

                fill_curves_filepath = join(sections_directory, f"fill_curve_s{k}.txt")

                np.savetxt(fill_curves_filepath, data, header = "h, area, wetted_perimeter")

        except AssertionError:
            print(f"not all files exist yet for index {idx}")
            pass 